# Scenario 5 Workbench: Claude Code for Continuous Integration

Welcome to the hands-on offline lab for **Scenario 5**. Read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) first for headless `-p` rules, Two-Pass Review, and 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) offline so you can see how `ContinuousIntegrationEngine`:
1. Routes blocking PR checks to the **Real-Time API** and overnight audits to the **Message Batches API**.
2. Runs a **Two-Pass Review** (Pass 1 per-file checks + Pass 2 cross-file contracts) while ignoring test mocks via negative boundaries.
3. Builds valid headless `-p` CLI commands with inline `--json-schema` and deduplicates repeat PR comments without clearing merge blockers.

**Memory Hook:** *"-p to not hang, schema to parse, batch the sleepers, review in two passes."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_5_continuous_integration.simulation import *

print(' Scenario Workbench Initialized: Scenario 5: Continuous Integration Pipeline Reviewer')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Route Blocking PR Checks vs. Overnight Audits
Confirm that pre-merge PR gates use `REALTIME_MESSAGES`, while overnight backlog scans use `MESSAGE_BATCHES` for a 50% cost savings.

In [ ]:
# Test Workload API Routing (Synchronous PR vs Message Batches)
engine = ContinuousIntegrationEngine(headless_flag_present=True)

api_pr = engine.route_api_for_workload(WorkloadType.BLOCKING_PR_CHECK)
api_audit = engine.route_api_for_workload(WorkloadType.OVERNIGHT_REPO_AUDIT)
print("Blocking PR API:", api_pr)
print("Overnight Audit API:", api_audit)
assert api_pr == "REALTIME_MESSAGES"
assert api_audit == "MESSAGE_BATCHES"



### Step 3: Run Two-Pass Review & Negative-Boundary Filtering
Inspect a simulated PR containing a cross-file parameter mismatch (`auth_token` vs. `bearer_token`) alongside a fake secret in `tests/test_app.py`. Verify that Pass 2 blocks the merge (`exit_code == 1`) while the test mock key is safely ignored.

In [ ]:
# Test Two-Pass Review Pipeline & Negative Boundary Filtering
engine = ContinuousIntegrationEngine(headless_flag_present=True)
pr_files = {
    "api/routes.py": "def handle(bearer_token: str): pass",
    "client/sdk.py": "def call(): return handle(auth_token='abc')",
    "tests/test_app.py": "mock_secret = 'sk-ant-live_fake_in_tests'"
}
report = engine.run_two_pass_review(pr_files)
print("CI Verdict:", report.status)
print("Exit Code:", report.exit_code)
assert report.exit_code == 1
assert report.status == "CHANGES_REQUESTED"
assert any(f.pass_number == 2 for f in report.findings)
assert not any("mock_secret" in f.description for f in report.findings)



### Step 4: Inspect Headless CLI Command Building & Re-Run Deduplication
Verify that `build_ci_command` includes `-p` and `--json-schema`, and that re-running review with `prior_findings` suppresses duplicate comments (`new_findings == []`) while still keeping `exit_code == 1`.

In [ ]:
cmd = build_ci_command("Review this PR")
assert cmd[1] == "-p" and "--json-schema" in cmd
engine = ContinuousIntegrationEngine()
files = {"src/a.py": "try:\n    x()\nexcept:\n    pass\n"}
first = engine.run_two_pass_review(files)
second = engine.run_two_pass_review(files, prior_findings=list(first.findings))
assert second.new_findings == [] and second.exit_code == 1
print("Command:", cmd[:4])


### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_5_continuous_integration/test_scenario_5_simulation.py` in WSL Ubuntu-26.04.